<div style="background:#03045E;padding:28px 32px;border-radius:16px;font-family:Arial,sans-serif;">
<img src="../assets/jekacode-logo.png" alt="Jekacode" width="240"/>
<p style="color:#16D365;font-size:12px;letter-spacing:2.5px;margin:18px 0 6px 0;">JEKACODE AI ENGINEERING · WEEK 3 · DAY 1</p>
<h1 style="color:#ffffff;margin:0;font-size:28px;">APIs, tokens, latency, limits</h1>
<p style="color:#d7deea;margin:10px 0 0 0;font-size:16px;">Gemini and Grok are kitchens. Your key is the ticket.</p>
</div>


## What we want to achieve

Explain API, key, token, context window, latency, hallucination, inconsistency. Call Gemini (and Grok if you have a key).

## Tools you need (names only)

`.env` with GEMINI_API_KEY. Optional GROK_API_KEY. Optional Ollama.

## How to (do these before the first code cell if you have not)

1. Bookmark [../guides/HOW_TO.md](../guides/HOW_TO.md)
2. VS Code open on the **course folder** · terminal shows `(.venv)`
3. Ollama: [https://ollama.com](https://ollama.com) then `ollama run llama3.2`
4. Gemini key: [Google AI Studio](https://aistudio.google.com/app/apikey) → copy `.env.example` to `.env` → `GEMINI_API_KEY=`
5. Grok key (optional): [console.x.ai](https://console.x.ai) → `GROK_API_KEY=`
6. This notebook: top right **Select Kernel** → Python in `.venv`

## What goes on behind the scenes

Your Python uses **requests** to POST JSON to a URL (**endpoint**). The server runs **inference** and returns tokens. You pay for tokens and you wait (**latency**). Nothing magical happens on your laptop except Ollama.

**Keys & installs (bookmark):** [../guides/HOW_TO.md](../guides/HOW_TO.md) · **Terms:** [../guides/AI_ENGINEERING_TERMS.md](../guides/AI_ENGINEERING_TERMS.md)


In [2]:
# --- Why this cell exists (read once) ---
# Python only finds packages that live on a list of folders called sys.path.
# This notebook sits in a week folder. The jekacode helper lives one folder up.
# Novices: you are not "hacking". You are telling Python where the course lives.

import sys
# sys = the "system" module. We use it to change where Python looks for imports.

from pathlib import Path
# Path is a friendly way to talk about folders. It works on Mac, Windows, and Linux.

root = Path.cwd()
# cwd = current working directory = "the folder this notebook thinks it is in".

if not (root / "jekacode").exists():
    # If we cannot see the jekacode folder here, we are inside week01, week02, ...
    root = root.parent
    # parent = the folder above this one (the course root).

if str(root) not in sys.path:
    sys.path.append(str(root))
    # Now `from jekacode.ai import ask` can succeed.

print("Course folder Python will use:", root)
print("You should see jekacode inside that folder.")


Course folder Python will use: c:\Users\Lenovo\Desktop\jekacode-ai-engineering
You should see jekacode inside that folder.


Pictures: [../visuals/api-architecture.html](../visuals/api-architecture.html) · [../visuals/latency.html](../visuals/latency.html)

| Import | Why |
|---|---|
| `requests` (inside jekacode.ai) | Speak HTTP, the language of APIs |
| `dotenv` | Load `.env` so keys are not in GitHub |
| `ask` | One function so you do not copy URLs every week |

GPT/Claude: **names to recognise**. We do not require those keys.

## Theory: tokens, context, cost, limits

- A **token** is a chip of text (often ~¾ of an English word). You pay for input + output tokens.
- **Context window** = how much the model can “see” at once. Paste a whole textbook → overflow or dropped start. That is why **RAG** (Week 6) sends *chunks*, not the library.
- **Rate limit** = too many requests; wait. Classroom 429 errors are normal if thirty people hit Gemini together.
- **Hallucination** vs **inconsistency**: wrong-but-confident vs same-prompt-different-wording. Test both with `ask_timed` on Day 2.


In [1]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 6.5244,     # Lagos
    "longitude": 3.3792,
    "current_weather": True,
}

response = requests.get(url, params=params)
data = response.json()      # JSON text → Python dict

print(data["current_weather"])


{'time': '2026-09-08T13:45', 'interval': 900, 'temperature': 28.4, 'windspeed': 11.4, 'winddirection': 195, 'is_day': 1, 'weathercode': 3}


In [ ]:
import os
import requests

def ask_gemini(prompt: str) -> str:
    """Send a prompt to Gemini and return the model's text reply."""
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Set GEMINI_API_KEY in your .env file first.")

    url = ("https://generativelanguage.googleapis.com/v1beta/models/"
           "gemini-1.5-flash:generateContent")
    payload = {"contents": [{"parts": [{"text": prompt}]}]}

    response = requests.post(f"{url}?key={api_key}", json=payload)
    response.raise_for_status()   # raises an error on a bad response

    result = response.json()
    return result["candidates"][0]["content"]["parts"][0]["text"]


reply = ask_gemini("Explain APIs to a beginner in one sentence.")
print(reply)